In [ ]:
from pathlib import Path
project_root=Path("../")
project_root

In [ ]:
# Data directoies
RAW_DATA = project_root / "data/raw"
CLEANED_DATA = project_root / "data/cleaned"
FILTERED_DATA = project_root / "data/filtered"
DEDUPED_DATA = project_root / "data/deduped"
PII_DATA = project_root / "data/pii"
FINAL_DATA = project_root / "data/final"
MANIFESTS = project_root / "manifests"

# create directories if they don't exist
for directory in [RAW_DATA, CLEANED_DATA, FILTERED_DATA, DEDUPED_DATA, PII_DATA, FINAL_DATA]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Raw data directory: {RAW_DATA.resolve()}")
print(f"Cleaned data directory: {CLEANED_DATA.resolve()}")
print(f"Filtered data directory: {FILTERED_DATA.resolve()}")
print(f"Deduped data directory: {DEDUPED_DATA.resolve()}")  
print(f"PII data directory: {PII_DATA.resolve()}")
print(f"Final data directory: {FINAL_DATA.resolve()}")

In [ ]:
from datasets import load_dataset
# Load the dataset from the Hugging Face Hub

uz_wiki = load_dataset("yakhyo/uz-wiki", split="train")



In [ ]:
uz_wiki

In [ ]:
type(uz_wiki)

In [ ]:
uz_wiki[0]

In [ ]:
uz_wiki[1]

In [ ]:
len(uz_wiki)

In [ ]:
uz_wiki.to_json(f"{RAW_DATA}/uz_wiki.jsonl", force_ascii=False)

In [ ]:
import json

documents = []
with open(f"{RAW_DATA}/uz_wiki.jsonl", encoding="utf-8") as f:
    for line in f:                       # read the file line by line
        documents.append(json.loads(line))  # turn one line into a dict

print(len(documents))
print(documents[1]["title"])

In [ ]:
raw_chars = sum(len(doc["text"]) for doc in documents)

print("Raw documents:", len(documents))
print("Raw characters:", raw_chars)
print("Raw size MB:", round(Path(f"{RAW_DATA}/uz_wiki.jsonl").stat().st_size / (1024**2), 2))

In [ ]:
documents[10]["text"]


In [ ]:
import html
import re
import unicodedata

def remove_html(text):
    return re.sub(r"<[^>]+>", " ", text)

def decode_entities(text):
    return html.unescape(text)

def normalize_unicode(text):
    return unicodedata.normalize("NFKC", text)

APOSTROPHES = ["ʻ", "ʼ", "‘", "’", "`", "´"]

def normalize_apostrophes(text):
    for symbol in APOSTROPHES:
        text = text.replace(symbol, "'")
    return text

def remove_control_chars(text):
    return "".join(
        ch for ch in text
        if ch in "\n\t" or unicodedata.category(ch)[0] != "C"
    )

def normalize_whitespace(text):
    text = re.sub(r"[ \t]+", " ", text)    # many spaces -> one space
    text = re.sub(r" *\n *", "\n", text)    # spaces around newlines
    text = re.sub(r"\n{3,}", "\n\n", text)  # 3+ newlines -> 2
    return text.strip()                     # trim start and end

def clean_text(text):
    text = remove_html(text)
    text = decode_entities(text)
    text = normalize_unicode(text)
    text = normalize_apostrophes(text)
    text = remove_control_chars(text)
    text = normalize_whitespace(text)
    return text

In [ ]:
documents[11]["text"][:500]

In [ ]:
clean_text(documents[11]["text"])[:500]

In [ ]:
cleaned = []
for doc in documents:
    text = clean_text(doc["text"])
    if text:                      # 7. empty-document removal
        cleaned.append({**doc, "text": text})

print("Before:", len(documents))
print("After:", len(cleaned))
print("Empty removed:", len(documents) - len(cleaned))

In [ ]:
changed = sum(
    1 for old, new in zip(documents, cleaned)
    if old["text"] != new["text"]
)
print("Changed documents:", changed)

In [ ]:
chars_before = sum(len(doc["text"]) for doc in documents)
chars_after = sum(len(doc["text"]) for doc in cleaned)

print("Characters before:", chars_before)
print("Characters after:", chars_after)
print(f"Saved: {chars_before - chars_after} ({(chars_before - chars_after) / chars_before:.2%})")

In [ ]:
before = documents[1]["text"][:200]
after = cleaned[1]["text"][:200]
print("BEFORE:", repr(before))
print("AFTER: ", repr(after))

In [ ]:
cleaned_file = CLEANED_DATA / "uz_wiki_cleaned.jsonl"

with open(cleaned_file, "w", encoding="utf-8") as f:
    for doc in cleaned:
        f.write(
            json.dumps(doc, ensure_ascii=False) + "\n"
        )

print(f"Saved cleaned data to: {cleaned_file}")

In [ ]:

manifest = {
    "source": "uz_wiki",
    "step": "cleaning",
    "input": f"{RAW_DATA.resolve()}/uz_wiki.jsonl",
    "output": f"{CLEANED_DATA.resolve()}/uz_wiki_cleaned.jsonl",
    "docs_before": len(documents),
    "docs_after": len(cleaned),
    "empty_removed": len(documents) - len(cleaned),
    "chars_before": chars_before,
    "chars_after": chars_after,
}

with open(f"{MANIFESTS}/uz_wiki_cleaned.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

# Filtering

In [ ]:
import json

cleaned_docs = []
with open(f"{CLEANED_DATA}/uz_wiki_cleaned.jsonl", encoding="utf-8") as f:
    for line in f:
        cleaned_docs.append(json.loads(line))

print("Loaded:", len(cleaned_docs))

In [ ]:
documents[100]

In [ ]:
cleaned_docs[100]
# print(cleaned_docs[100]["text"])

In [ ]:
lengths = [len(doc["text"]) for doc in cleaned_docs]

print("Shortest:", min(lengths))
print("Longest:", max(lengths))
print("Average:", sum(lengths) // len(lengths))

short = [n for n in lengths if n < 200]
print("Texts shorter than 200 characters:", len(short))

In [ ]:
total = len(lengths)
print("Total documents:", total)
print("Short (<200):", len(short), f"({len(short) / total:.1%})")

In [ ]:
buckets = [0, 50, 100, 200, 500, 1000, 5000, 200000]

for low, high in zip(buckets, buckets[1:]):
    count = sum(1 for n in lengths if low <= n < high)
    print(f"{low:>6} - {high:<6}: {count:>7} ({count / total:.1%})")

In [ ]:

import random

random.seed(42)
for low, high in [(0, 50), (50, 100), (100, 200), (200, 500)]:
    group = [doc for doc in cleaned_docs if low <= len(doc["text"]) < high]
    print(f"\n===== {low}-{high} characters =====")
    for doc in random.sample(group, 3):
        print(doc["title"], "->", repr(doc["text"][:150]))

In [ ]:
FILTER_CONFIG = {
    "min_chars": 200,
    "min_alpha_ratio": 0.50,
    "max_repeated_line_ratio": 0.30
}

FILTER_CONFIG

In [ ]:
from collections import Counter

def alphabetic_ratio(text):
    """
    Percentage of characters that are alphabetic.
    """
    if not text:
        return 0.0

    alpha_count = sum(char.isalpha() for char in text)

    return alpha_count / len(text)

def repeated_line_ratio(text):
    """Share of non-empty lines that appear more than once."""
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if len(lines) <= 1:
        return 0.0
    counts = Counter(lines)
    repeated = sum(c for c in counts.values() if c > 1)
    return repeated / len(lines)

In [ ]:
filtered_docs = []
removed_docs = []  

filter_stats = {
    "input": len(cleaned_docs),
    "min_chars": 0,
    "alpha_ratio": 0,
    "repeated_lines": 0,
    "output": 0
}

for doc in cleaned_docs:
    text = doc["text"]

    # Mimimum character length
    if len(text) < FILTER_CONFIG["min_chars"]:
        doc["filtered"] = True
        doc["filter_reason"] = "min_chars"
        filter_stats["min_chars"] += 1
        removed_docs.append(doc)
        continue

    # Alphabetic ratio
    alpha_ratio = alphabetic_ratio(text)
    if alpha_ratio < FILTER_CONFIG["min_alpha_ratio"]:
        doc["filtered"] = True
        doc["filter_reason"] = "min_alpha_ratio"
        filter_stats["alpha_ratio"] += 1
        removed_docs.append(doc)
        continue

    # Repeated line ratio
    repeated_ratio = repeated_line_ratio(text)
    if repeated_ratio > FILTER_CONFIG["max_repeated_line_ratio"]:
        doc["filtered"] = True
        doc["filter_reason"] = "max_repeated_line_ratio"
        filter_stats["repeated_lines"] += 1
        removed_docs.append(doc)
        continue

    # If we made it here, the document is not filtered
    filtered_docs.append(doc)

filter_stats["output"] = len(filtered_docs)

print("Filter stats:", filter_stats)

In [ ]:
removed_docs

In [ ]:
x = [doc for doc in removed_docs if doc["filter_reason"] == "max_repeated_line_ratio"]

In [ ]:
print(x[0]["text"])


In [ ]:
for doc in removed_docs:
    if doc["filter_reason"]:
        print(doc["title"], "->", repr(doc["text"][:150]))

In [ ]:
removed_docs

In [ ]:
# Save documents that passed the filters
filtered_file = FILTERED_DATA / "uz_wiki_filtered.jsonl"

with open(filtered_file, "w", encoding="utf-8") as f:
    for doc in filtered_docs:
        f.write(json.dumps(doc, ensure_ascii=False) + "\n")

print(f"Saved filtered data to: {filtered_file}")

In [ ]:
#  Manifest
manifest = {
    "source": "uz_wiki",
    "step": "filtering",
    "input": f"{CLEANED_DATA.resolve()}/uz_wiki_cleaned.jsonl",
    "output": f"{FILTERED_DATA.resolve()}/uz_wiki_filtered.jsonl",
    "config": FILTER_CONFIG,
    "docs_before": filter_stats["input"],
    "docs_after": filter_stats["output"],
    "removed_total": filter_stats["input"] - filter_stats["output"],
    "removed_by_reason": {
        "min_chars": filter_stats["min_chars"],
        "min_alpha_ratio": filter_stats["alpha_ratio"],
        "max_repeated_line_ratio": filter_stats["repeated_lines"],
    },
}

with open(MANIFESTS / "uz_wiki_filtered.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("Manifest saved.")

# Deduplication

In [ ]:
import hashlib
import re

def dedup_key(text):
    """Simplified version of the text, used only for comparison."""
    text = text.lower()                 # ignore upper/lower case
    text = re.sub(r"\s+", " ", text)    # all spaces and newlines -> one space
    return text.strip()

In [ ]:
def text_hash(text):
    key = dedup_key(text)
    return hashlib.sha256(key.encode("utf-8")).hexdigest()

In [ ]:
print(text_hash("Toshkent — poytaxt."))
print(text_hash("toshkent —  POYTAXT."))   # same hash
print(text_hash("Samarqand — qadimiy shahar."))   # different hash

In [ ]:
first_title = {}          # hash -> title of the first document with this text
deduped_docs = []
duplicate_docs = []

for doc in filtered_docs:
    h = text_hash(doc["text"])
    if h in first_title:
        duplicate_docs.append({**doc, "duplicate_of": first_title[h]})
        continue
    first_title[h] = doc["title"]
    deduped_docs.append(doc)

print("Duplicates removed:", len(duplicate_docs))
for doc in duplicate_docs[:5]: print(doc["title"], "== duplicate of ==>", doc["duplicate_of"])

In [ ]:
by_title = {doc["title"]: doc for doc in filtered_docs}

a = by_title["Gʻoyibon"]["text"]
b = by_title["Oqmachit (shaharcha)"]["text"]

print(a[:300])
print("-----")
print(b[:300])
print("Identical after normalization:", dedup_key(a) == dedup_key(b))

In [ ]:
deduped_file = DEDUPED_DATA / "uz_wiki_deduped.jsonl"

with open(deduped_file, "w", encoding="utf-8") as f:
    for doc in deduped_docs:
        f.write(json.dumps(doc, ensure_ascii=False) + "\n")

manifest = {
    "source": "uz_wiki",
    "step": "exact_dedup",
    "method": "SHA-256 of lowercased, whitespace-normalized text; first occurrence kept",
    "input": f"{FILTERED_DATA.resolve()}/uz_wiki_filtered.jsonl",
    "output": f"{DEDUPED_DATA.resolve()}/uz_wiki_deduped.jsonl",
    "docs_before": len(filtered_docs),
    "docs_after": len(deduped_docs),
    "duplicates_removed": len(duplicate_docs),
    "examples": [
        {"removed": d["title"], "duplicate_of": d["duplicate_of"]}
        for d in duplicate_docs[:5]
    ],
}

with open(MANIFESTS / "uz_wiki_deduped.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("Saved:", deduped_file)

# PII

In [ ]:
import re

PII_PATTERNS = {
    "EMAIL": re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b"),
    "PHONE": re.compile(
        r"(?:\+998[\s-]?\d{2}[\s-]?\d{3}[\s-]?\d{2}[\s-]?\d{2})"   # Uzbekistan
        r"|(?:\+82[\s-]?\d{1,2}[\s-]?\d{3,4}[\s-]?\d{4})"          # Korea, international
        r"|(?:\b01[016789][\s-]?\d{3,4}[\s-]?\d{4}\b)"              # Korea, mobile
    ),
    "IP": re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b"),
}

In [ ]:
def mask_pii(text):
    counts = {}
    for label, pattern in PII_PATTERNS.items():
        text, n = pattern.subn(f"<{label}>", text)
        counts[label] = n
    return text, counts

sample = "Murojaat: info@sunbae.uz yoki +998 90 123 45 67. Koreyada: 010-1234-5678. Server 192.168.0.1. Aholisi 3810 nafar (2010)."
masked, counts = mask_pii(sample)
print(masked)
print(counts)

In [ ]:
for label, pattern in PII_PATTERNS.items():
    found = []
    for doc in deduped_docs:
        for match in pattern.finditer(doc["text"]):
            found.append((doc["title"], match.group()))
    print(f"\n===== {label}: {len(found)} matches =====")
    for title, value in found[:10]:
        print(f"  {title} -> {value}")

In [ ]:
from collections import Counter

pii_docs = []
pii_stats = Counter()
docs_with_pii = 0

for doc in deduped_docs:
    text, counts = mask_pii(doc["text"])
    pii_stats.update(counts)
    if sum(counts.values()) > 0:
        docs_with_pii += 1
    pii_docs.append({**doc, "text": text})

print("Documents with PII:", docs_with_pii)
print("Masked:", dict(pii_stats))

In [ ]:
pii_file = PII_DATA / "uz_wiki_pii.jsonl"

with open(pii_file, "w", encoding="utf-8") as f:
    for doc in pii_docs:
        f.write(json.dumps(doc, ensure_ascii=False) + "\n")

manifest = {
    "source": "uz_wiki",
    "step": "pii_masking",
    "method": "regex masking: EMAIL, PHONE (+998, +82, Korean mobile), IP",
    "not_masked": "person names (public figures in Wikipedia)",
    "input": f"{DEDUPED_DATA.resolve()}/uz_wiki_deduped.jsonl",
    "output": f"{PII_DATA.resolve()}/uz_wiki_pii.jsonl",
    "docs": len(pii_docs),
    "docs_with_pii": docs_with_pii,
    "masked_counts": dict(pii_stats),
}

with open(MANIFESTS / "uz_wiki_pii.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

# Final statistics

In [ ]:
import json

steps = {
    "cleaning":    MANIFESTS / "uz_wiki_cleaned.json",
    "filtering":   MANIFESTS / "uz_wiki_filtered.json",
    "exact_dedup": MANIFESTS / "uz_wiki_deduped.json",
    "pii_masking": MANIFESTS / "uz_wiki_pii.json",
}

manifests = {}
for name, path in steps.items():
    with open(path, encoding="utf-8") as f:
        manifests[name] = json.load(f)
    print("Loaded:", name)

In [ ]:
raw_docs = manifests["cleaning"]["docs_before"]

rows = [
    ("Raw",          raw_docs,                                    0),
    ("Cleaning",     manifests["cleaning"]["docs_after"],         manifests["cleaning"]["empty_removed"]),
    ("Filtering",    manifests["filtering"]["docs_after"],        manifests["filtering"]["removed_total"]),
    ("Exact dedup",  manifests["exact_dedup"]["docs_after"],      manifests["exact_dedup"]["duplicates_removed"]),
    ("PII masking",  manifests["pii_masking"]["docs"],            0),
]

print(f"{'Step':<14}{'Docs after':>12}{'Removed':>10}{'Kept of raw':>13}")
for step, docs_after, removed in rows:
    print(f"{step:<14}{docs_after:>12,}{removed:>10,}{docs_after / raw_docs:>13.1%}")

In [ ]:
print("\nFiltering - removed by reason:")
for reason, count in manifests["filtering"]["removed_by_reason"].items():
    print(f"  {reason:<28}{count:>8,}")

print("\nPII masking:")
print(f"  documents with PII{manifests['pii_masking']['docs_with_pii']:>18,}")
for label, count in manifests["pii_masking"]["masked_counts"].items():
    print(f"  {label:<28}{count:>8,}")

In [ ]:
final_docs = []
with open(PII_DATA / "uz_wiki_pii.jsonl", encoding="utf-8") as f:
    for line in f:
        final_docs.append(json.loads(line))

total_chars = sum(len(d["text"]) for d in final_docs)
total_words = sum(len(d["text"].split()) for d in final_docs)

print("\nFinal dataset:")
print(f"  documents        {len(final_docs):>14,}")
print(f"  characters       {total_chars:>14,}")
print(f"  words            {total_words:>14,}")
print(f"  avg chars / doc  {total_chars // len(final_docs):>14,}")

In [ ]:
summary = {
    "source": "uz_wiki",
    "pipeline": list(steps.keys()),
    "raw_docs": raw_docs,
    "final_docs": len(final_docs),
    "kept_ratio": round(len(final_docs) / raw_docs, 4),
    "removed": {
        "empty_after_cleaning": manifests["cleaning"]["empty_removed"],
        **manifests["filtering"]["removed_by_reason"],
        "exact_duplicates": manifests["exact_dedup"]["duplicates_removed"],
    },
    "pii_masked": manifests["pii_masking"]["masked_counts"],
    "final_size": {"characters": total_chars, "words": total_words},
}

with open(MANIFESTS / "uz_wiki_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\nSummary saved.")